# Handwritten Character Recognition using CNN

**Objective:** Recognize handwritten digits (0–9) from images using image processing and a Convolutional Neural Network (CNN).

**Dataset:** The runnable demo uses scikit-learn's built-in digits dataset (1,797 images, 8×8 grayscale). An optional MNIST section is included for 28×28 digits.

**Scope note:** This starter project recognizes digits, not every alphabet or whole words. EMNIST letters and CRNN word recognition are future extensions.

## 1. Import libraries

In [ ]:
# Install dependencies in VS Code terminal: pip install -r requirements.txt
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.utils import to_categorical

RANDOM_STATE = 42

## 2. Load and prepare images

In [ ]:
digits = load_digits()
X = digits.images.astype("float32") / 16.0
y = digits.target.astype("int64")
X = X[..., np.newaxis]
y_cat = to_categorical(y, num_classes=10)

print("Images shape:", X.shape)
print("Labels shape:", y.shape)

## 3. View sample images

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(X[i].squeeze(), cmap="gray")
    ax.set_title(f"Label: {y[i]}")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. Train/test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_cat, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print("Training images:", len(X_train))
print("Testing images:", len(X_test))

## 5. Build CNN

In [ ]:
model = Sequential([
    Conv2D(32, (3, 3), activation="relu", padding="same", input_shape=X_train.shape[1:]),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(64, activation="relu"),
    Dropout(0.25),
    Dense(10, activation="softmax")
])
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()

## 6. Train model

In [ ]:
history = model.fit(
    X_train, y_train, validation_split=0.1,
    epochs=12, batch_size=32, verbose=1
)

## 7. Evaluate

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
y_true = np.argmax(y_test, axis=1)
y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
print(f"Test accuracy: {test_accuracy:.4f}")
print(classification_report(y_true, y_pred, digits=4))

## 8. Training accuracy plot

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history.history["accuracy"], label="Training accuracy")
plt.plot(history.history["val_accuracy"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("CNN Training Accuracy")
plt.legend()
plt.tight_layout()
plt.show()

## 9. Confusion matrix

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(y_true, y_pred, ax=ax)
ax.set_title("Handwritten Digit Recognition - Confusion Matrix")
plt.tight_layout()
plt.show()

## 10. Predict a test image

In [ ]:
index = 0
plt.imshow(X_test[index].squeeze(), cmap="gray")
plt.title(f"Actual: {y_true[index]} | Predicted: {y_pred[index]}")
plt.axis("off")
plt.show()
print("Predicted digit:", y_pred[index])

## 11. Optional full MNIST dataset (28×28)

To use MNIST instead, replace the data-loading and split cells with the following code. TensorFlow downloads MNIST on the first run, so internet may be required.

```python
from tensorflow.keras.datasets import mnist
(X_train, y_train), (X_test, y_test) = mnist.load_data()
X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0
X_train = X_train[..., np.newaxis]
X_test = X_test[..., np.newaxis]
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)
```

The CNN can then be trained on the MNIST arrays.

## Conclusion
The project demonstrates image preparation, CNN training, evaluation, visualization, and prediction. Recognizing alphabet characters requires a dataset such as EMNIST Letters; recognizing full words or sentences requires a sequence-recognition model such as CRNN.